# GCE Eligibility - 01: Data Prep

Loads `pluto_residential_clean.csv` from Unity Catalog Volume, preprocesses, and writes train/val/test splits to the same Volume for downstream notebooks.

In [ ]:
# verify files are present before proceeding
import os
VOL = "/Volumes/ml_final_workspace/default/ml_final"
for f in os.listdir(VOL):
    size = os.path.getsize(f"{VOL}/{f}")
    print(f, f"{size/1e6:.1f} MB")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

VOL = "/Volumes/ml_final_workspace/default/ml_final"

# read with Spark — demonstrates Spark ingestion for portfolio context
sdf = spark.read.csv(
    f"dbfs:{VOL}/pluto_residential_clean.csv",
    header=True,
    inferSchema=True
)
print(f"Spark DataFrame: {sdf.count():,} rows, {len(sdf.columns)} columns")

# convert to pandas — sklearn and torch operate on pandas/numpy
df = sdf.toPandas()
print(df['gce_eligible'].value_counts())


In [ ]:
# borough eligibility breakdown — show as Spark DataFrame for portfolio signal
import pyspark.sql.functions as F

elig_by_borough = (
    sdf.groupBy("Borough")
       .agg(
           F.count("*").alias("total_parcels"),
           F.sum(F.col("gce_eligible").cast("int")).alias("eligible_parcels")
       )
       .withColumn("eligibility_rate", F.round(
           F.col("eligible_parcels") / F.col("total_parcels"), 4
       ))
       .orderBy(F.desc("eligibility_rate"))
)
elig_by_borough.show()


In [ ]:
# feature selection — UnitsRes and YearBuilt withheld (define the eligibility rule)
feature_cols = ['NumFloors', 'BldgArea', 'ResArea', 'LotArea', 'AssessLand',
                'AssessTot', 'ExemptTot', 'NumBldgs', 'Latitude', 'Longitude', 'Borough']

X_raw = df[feature_cols].copy()
y     = df['gce_eligible'].astype(int).copy()

# drop nulls — 0.7% loss, preserves feature integrity vs imputation artifacts
X_raw = X_raw.dropna()
y     = y[X_raw.index]

# one-hot encode Borough — nominal category, no ordinal relationship
# drop_first=True leaves Brooklyn as implicit reference category
X_encoded = pd.get_dummies(X_raw, columns=['Borough'], drop_first=True)

print(f"Shape after null drop and encoding: {X_encoded.shape}")
print(f"Columns: {X_encoded.columns.tolist()}")


In [ ]:
# StandardScaler on continuous features only
# borough one-hot columns excluded — scaling binary indicators destroys their meaning
# StandardScaler preferred over MinMaxScaler: robust to the extreme-valued distributions in NYC property data
continuous_cols = ['NumFloors', 'BldgArea', 'ResArea', 'LotArea', 'AssessLand',
                   'AssessTot', 'ExemptTot', 'NumBldgs', 'Latitude', 'Longitude']

scaler    = StandardScaler()
X_scaled  = X_encoded.copy()
X_scaled[continuous_cols] = scaler.fit_transform(X_encoded[continuous_cols])

print(X_scaled[continuous_cols].describe().round(3))


In [ ]:
# stratified split — preserves 1:19 class ratio across all three partitions
# random_state=42 matches ml_collab.py exactly for reproducibility
X = X_scaled.values.astype(np.float32)
# y already aligned to X_raw.index (post-dropna) above — use .values directly
y_arr = y.values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_arr, test_size=0.30, random_state=42, stratify=y_arr)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape}, pos={y_train.sum():,}")
print(f"Val:   {X_val.shape},   pos={y_val.sum():,}")
print(f"Test:  {X_test.shape},  pos={y_test.sum():,}")


In [ ]:
import os, pickle

VOL = "/Volumes/ml_final_workspace/default/ml_final"
SPLITS = f"{VOL}/splits"
os.makedirs(SPLITS, exist_ok=True)

np.save(f"{SPLITS}/X_train.npy", X_train)
np.save(f"{SPLITS}/X_val.npy",   X_val)
np.save(f"{SPLITS}/X_test.npy",  X_test)
np.save(f"{SPLITS}/y_train.npy", y_train)
np.save(f"{SPLITS}/y_val.npy",   y_val)
np.save(f"{SPLITS}/y_test.npy",  y_test)

with open(f"{SPLITS}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Splits and scaler written to Volume.")
for fname in os.listdir(SPLITS):
    size = os.path.getsize(f"{SPLITS}/{fname}")
    print(f"  {fname}: {size/1e6:.1f} MB")
